# Multithreading, Multiprocessing & Parallel Computing

## 1. Threading Basics

In [ ]:
#basic thread
import threading
import time

def display():
    for i in range(3):
        time.sleep(1)
        print("thread started")

t1 = threading.Thread(target=display)
t1.start()
t1.join()

In [ ]:
#thread passing values
import threading
import time

def display(x, delay):
    for i in range(x):
        time.sleep(delay)
        print("thread started")

t1 = threading.Thread(target=display, args=(3, 1))
t1.start()
t1.join()

In [ ]:
#multiple threads
import threading
import time

def display(x, s, name):
    for i in range(x):
        time.sleep(s)
        print(name, "::started")

t1 = threading.Thread(target=display, args=(3, 1, "thread1"))
t2 = threading.Thread(target=display, args=(3, 0.5, "thread2"))
t1.start()
t2.start()
t1.join()
t2.join()

## 2. Thread Naming, State & Lifecycle

In [ ]:
#naming threads
import threading
import time

def display(x):
    for i in range(3):
        time.sleep(0.5)
        print(threading.current_thread().name, "-> thread started")

for p in range(3):
    t = threading.Thread(target=display, args=(p,))
    t.name = f"thread#{p}"
    t.start()

In [ ]:
#is_alive(), ctime()
import threading
import time

def display(i):
    time.sleep(i)

t1 = threading.Thread(target=display, args=(3,), name="Thread#1")
t1.start()
t2 = threading.Thread(target=display, args=(1,), name="Thread#2")
t2.start()

for x in range(4):
    time.sleep(1)
    print('[', time.ctime(), t1.name, t1.is_alive(), ']')
    print('[', time.ctime(), t2.name, t2.is_alive(), ']')

In [ ]:
#threading.current_thread(), threading.enumerate(), threading.active_count()
import threading
import time

def work():
    time.sleep(2)

for i in range(3):
    threading.Thread(target=work).start()

print("enumerate:", threading.enumerate())
print("active_count:", threading.active_count())

## 3. Worker Pools & Daemon Threads

In [ ]:
#ThreadPoolExecutor - defining a pool of worker threads
import time
from concurrent.futures import ThreadPoolExecutor

def do_job():
    time.sleep(1)
    print("job done")

with ThreadPoolExecutor(max_workers=2) as executor:
    future1 = executor.submit(do_job)
    future2 = executor.submit(do_job)
    future1.result()
    future2.result()

In [ ]:
#threading.current_thread(), os.getpid()
import threading
import os

def task1():
    print(f"Task 1 on thread: {threading.current_thread().name}, pid: {os.getpid()}")

def task2():
    print(f"Task 2 on thread: {threading.current_thread().name}, pid: {os.getpid()}")

print(f"Main thread: {threading.current_thread().name}, pid: {os.getpid()}")

t1 = threading.Thread(target=task1, name="t1")
t2 = threading.Thread(target=task2, name="t2")
t1.start()
t2.start()
t1.join()
t2.join()

### Daemon Threads

A **daemon thread** (`daemon=True`) is a background thread tied to the lifetime of the main program. Once every *non-daemon* thread has finished, the interpreter exits immediately and kills any remaining daemon threads outright - it does not wait for them, and no cleanup code in them gets a chance to run.

**Use case:** "fire and forget" background work that should never hold up the program from exiting - e.g. periodic housekeeping, heartbeat/monitoring pings, log flushing, cache warmers. Anything where finishing the task isn't essential to a correct shutdown.

**Pros:**
- The program exits promptly instead of hanging on background work.
- No need to explicitly track and stop every helper thread yourself.

**Cons:**
- No graceful shutdown - the thread can be killed mid-operation, so it must never be holding a lock, writing a file/DB record, or doing anything that needs to complete safely.
- Because of that risk, daemon threads are the wrong tool whenever a clean stop matters - use the **Stoppable Threads** pattern (section 5 above) instead in that case.

The cell below shows this directly: `work_a` is a daemon thread sleeping for 2 seconds, `work_b` is a normal thread that finishes instantly. Both threads are `join()`ed here so you can see the daemon thread's "finished" message - but without that `join()`, the program could exit the moment `work_b` completes, cutting `work_a` off mid-sleep.

In [1]:
#daemon threads
import threading
import time

def work_a():
    print("Thread#1 (daemon) started")
    time.sleep(2)
    print("Thread#1 (daemon) finished")

def work_b():
    print("Thread#2 started")
    print("Thread#2 finished")

t1 = threading.Thread(target=work_a, daemon=True)
t1.start()
t2 = threading.Thread(target=work_b)
t2.start()
t1.join()
t2.join()

Thread#1 (daemon) started
Thread#2 started
Thread#2 finished
Thread#1 (daemon) finished


**A more convincing demo.** The cell above `join()`s both threads, so it never actually shows a daemon thread getting cut off mid-work - it just shows two threads finishing normally. The real defining behavior is: *when the process exits, any daemon thread is killed instantly, unfinished*.

Dropping the `.join()` above wouldn't show that either: "the process exits" means the whole Python process ends, but a Jupyter kernel stays alive between cells, so an un-joined daemon thread would just keep running quietly in the background instead of getting killed.

To see the real thing, spawn an actual short-lived **subprocess** and let it exit on its own - once with `daemon=True`, once with `daemon=False` - and compare what printed and how long each took.

In [2]:
import subprocess
import sys
import textwrap
import time

script = textwrap.dedent("""
    import threading, time, sys

    def worker():
        for i in range(5):
            print(f"daemon working... {{i}}", flush=True)
            time.sleep(1)
        print("daemon finished all 5 steps", flush=True)

    t = threading.Thread(target=worker, daemon={daemon_flag})
    t.start()

    print("main thread doing 2s of work", flush=True)
    time.sleep(2)
    print("main thread exiting", flush=True)
""")

for daemon_flag in (True, False):
    code = script.format(daemon_flag=daemon_flag)
    start = time.time()
    result = subprocess.run([sys.executable, "-c", code], capture_output=True, text=True)
    elapsed = time.time() - start
    print(f"--- daemon={daemon_flag}  (process ran for {elapsed:.1f}s) ---")
    print(result.stdout)

--- daemon=True  (process ran for 2.1s) ---
daemon working... 0
main thread doing 2s of work
daemon working... 1
main thread exiting

--- daemon=False  (process ran for 5.1s) ---
daemon working... 0
main thread doing 2s of work
daemon working... 1
main thread exiting
daemon working... 2
daemon working... 3
daemon working... 4
daemon finished all 5 steps



In [ ]:
#timer thread
import threading
import time

def display():
    print("welcome")

t = threading.Timer(2.0, display)
t.start()
time.sleep(3)
print("timer finished, is_alive:", t.is_alive())

## 4. Thread Synchronization

**Why the delay matters.** A plain `x += 1` runs in nanoseconds, and CPython only considers switching threads roughly every 5ms (`sys.getswitchinterval()`). So the odds of a switch landing exactly between reading and writing `x` are astronomically small - the unlocked version below would come out "correct" almost every time even with millions of iterations, which would make it look like locks don't matter. To actually expose the race, the increment is split into read → tiny `sleep` → write, which widens that window enough for the two threads to reliably collide.

In [1]:
#race condition - shared counter without a lock
import threading
import time

x = 0

def increment():
    global x
    temp = x
    time.sleep(0.0005)  # widen the window so a thread switch can land here
    x = temp + 1

def thread_task():
    for _ in range(300):
        increment()

t1 = threading.Thread(target=thread_task)
t2 = threading.Thread(target=thread_task)
t1.start()
t2.start()
t1.join()
t2.join()

print("x =", x, "(expected 600 - without a lock, updates get lost)")

x = 300 (expected 600 - without a lock, updates get lost)


In [2]:
#lock, acquire, release
import threading
import time

lock = threading.Lock()
counter = 0

def safe_increment():
    global counter
    for _ in range(300):
        with lock:
            temp = counter
            time.sleep(0.0005)
            counter = temp + 1

t1 = threading.Thread(target=safe_increment)
t2 = threading.Thread(target=safe_increment)
t1.start()
t2.start()
t1.join()
t2.join()

print("counter =", counter, "(always 600 - the lock keeps the read+write together)")

counter = 600 (always 600 - the lock keeps the read+write together)


In [ ]:
#RLock - a lock that can be re-acquired by the same thread
import threading

class Counter:
    def __init__(self):
        self.a = 5
        self.b = 10
        self.lock = threading.RLock()

    def first(self):
        with self.lock:
            self.a += 5

    def second(self):
        with self.lock:
            self.b -= 5

    def main(self):
        # main() already holds the lock; first()/second() re-acquire it here.
        # a plain Lock would deadlock in this situation - RLock allows re-entry.
        with self.lock:
            self.first()
            self.second()
            print(self.a, self.b)

Counter().main()

In [ ]:
#event object - set(), clear(), wait(), is_set()
import threading
import time

def is_set_worker(event):
    event.set()
    print("event is set")
    time.sleep(3)
    event.clear()
    print("event is clear")

def waiting_worker(event):
    event.wait()
    while event.is_set():
        time.sleep(0.5)
        print("thread is waiting for signal")
    print("thread received signal")

event = threading.Event()
t1 = threading.Thread(target=is_set_worker, args=(event,))
t2 = threading.Thread(target=waiting_worker, args=(event,))
t1.start()
t2.start()
t1.join()
t2.join()

## 5. Stoppable Threads

`threading.Thread` has no built-in way to force-stop it from outside (`t.kill()` doesn't exist - it raises `AttributeError`). The standard pattern is to give the thread a stop flag (a `threading.Event`) that it checks on every loop iteration, and to ask it to stop cooperatively instead of killing it.

In [ ]:
import threading
import time

class StoppableThread(threading.Thread):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._stop_event = threading.Event()

    def stop(self):
        self._stop_event.set()

    def stopped(self):
        return self._stop_event.is_set()

    def run(self):
        i = 0
        while not self.stopped():
            print("working...", i)
            i += 1
            time.sleep(0.5)
        print("thread stopped gracefully")

t = StoppableThread()
t.start()
time.sleep(2)
t.stop()
t.join()
print("is_alive:", t.is_alive())

## 6. Communication Between Threads (Queue)

`queue.Queue` is thread-safe and is the standard way for threads to hand data to each other, instead of sharing raw variables. Here one thread produces items and another consumes them; a `None` sentinel tells the consumer when to stop.

In [ ]:
import threading
import queue
import time

q = queue.Queue()

def producer():
    for i in range(5):
        item = f"item-{i}"
        q.put(item)
        print("produced", item)
        time.sleep(0.3)
    q.put(None)  # sentinel to tell the consumer to stop

def consumer():
    while True:
        item = q.get()
        if item is None:
            break
        print("consumed", item)
        q.task_done()

t1 = threading.Thread(target=producer)
t2 = threading.Thread(target=consumer)
t1.start()
t2.start()
t1.join()
t2.join()

## 7. Multiprocessing Basics

In [ ]:
#running multiple processes
import multiprocessing
import os

def print_square(num):
    print(f"Square: {num*num} (pid: {os.getpid()})")

def print_cube(num):
    print(f"Cube: {num**3} (pid: {os.getpid()})")

if __name__ == "__main__":
    print("Main process pid:", os.getpid())

    p1 = multiprocessing.Process(target=print_square, args=(10,))
    p2 = multiprocessing.Process(target=print_cube, args=(10,))
    p1.start()
    p2.start()
    p1.join()
    p2.join()
    print("processes done - note the different pids above")

In [ ]:
#processes vs threads - same work, compare the pids
import threading
import os

def print_square_t(num):
    print(f"Square: {num*num} (thread: {threading.current_thread().name}, pid: {os.getpid()})")

def print_cube_t(num):
    print(f"Cube: {num**3} (thread: {threading.current_thread().name}, pid: {os.getpid()})")

t1 = threading.Thread(target=print_square_t, args=(10,))
t2 = threading.Thread(target=print_cube_t, args=(10,))
t1.start()
t2.start()
t1.join()
t2.join()
print("threads done - note the SAME pid above, unlike the processes example")

## 8. Process Pool and Map

In [ ]:
#ThreadPool.map() - a pool of THREADS (still limited by the GIL for CPU work)
from multiprocessing.dummy import Pool as ThreadPool

def square_number(n):
    return n ** 2

with ThreadPool(4) as pool:
    results = pool.map(square_number, [1, 2, 3, 4, 5])

print(results)

In [ ]:
#multiprocessing.Pool.map() - a pool of real PROCESSES
import multiprocessing as mp

def cube(n):
    return n ** 3

if __name__ == "__main__":
    with mp.Pool(processes=4) as pool:
        results = pool.map(cube, [1, 2, 3, 4, 5])
    print(results, "(runs in separate processes, no GIL contention)")

## 9. Sharing State Between Processes

Unlike threads, separate processes don't share memory by default - each has its own copy of everything. To share state you need explicit constructs: `multiprocessing.Value`/`Array` for simple shared C-type data (with a built-in lock), or `multiprocessing.Manager` for shared Python objects like `dict`/`list`.

In [ ]:
#multiprocessing.Value and multiprocessing.Array
import multiprocessing as mp

def increment_value(shared_val, n):
    for _ in range(n):
        with shared_val.get_lock():
            shared_val.value += 1

def fill_array(shared_arr):
    for i in range(len(shared_arr)):
        shared_arr[i] = i * i

if __name__ == "__main__":
    shared_counter = mp.Value('i', 0)
    shared_array = mp.Array('i', 5)

    p1 = mp.Process(target=increment_value, args=(shared_counter, 100000))
    p2 = mp.Process(target=increment_value, args=(shared_counter, 100000))
    p3 = mp.Process(target=fill_array, args=(shared_array,))

    p1.start(); p2.start(); p3.start()
    p1.join(); p2.join(); p3.join()

    print("shared counter:", shared_counter.value, "(200000 - the built-in lock keeps it safe across processes)")
    print("shared array:", list(shared_array))

In [ ]:
#multiprocessing.Manager - shared dict/list across processes
import multiprocessing as mp

def worker(shared_dict, key, value):
    shared_dict[key] = value

if __name__ == "__main__":
    with mp.Manager() as manager:
        shared_dict = manager.dict()
        processes = [mp.Process(target=worker, args=(shared_dict, i, i * i)) for i in range(5)]
        for p in processes:
            p.start()
        for p in processes:
            p.join()
        print(dict(shared_dict))

## 10. Global Interpreter Lock (GIL)

CPython's GIL allows only one thread to execute Python bytecode at a time, even on a multi-core machine. This means:
- Threads are great for **I/O-bound** work (network calls, file I/O, `time.sleep`) - while one thread waits on I/O, it releases the GIL and another thread can run.
- Threads give little/no speedup for **CPU-bound** work, because only one thread runs Python code at any instant.
- `multiprocessing` sidesteps the GIL entirely - each process has its own interpreter and its own GIL, so CPU-bound work actually runs in parallel across cores.

The timings below make this concrete: 2 threads doing CPU-bound work barely beat 1, while 2 processes doing the same work show a real speedup.

In [ ]:
import time
import threading
import multiprocessing as mp

def cpu_bound(n):
    count = 0
    for i in range(n):
        count += i * i
    return count

N = 5_000_000

#1) serial baseline - twice the work, one core
start = time.time()
cpu_bound(N)
cpu_bound(N)
print("serial (1 core, 2x work):", round(time.time() - start, 2), "s")

#2) two threads doing the same CPU-bound work
start = time.time()
t1 = threading.Thread(target=cpu_bound, args=(N,))
t2 = threading.Thread(target=cpu_bound, args=(N,))
t1.start(); t2.start()
t1.join(); t2.join()
print("2 threads:", round(time.time() - start, 2), "s (barely faster - the GIL serializes CPU-bound bytecode)")

if __name__ == "__main__":
    #3) two processes doing the same CPU-bound work
    start = time.time()
    p1 = mp.Process(target=cpu_bound, args=(N,))
    p2 = mp.Process(target=cpu_bound, args=(N,))
    p1.start(); p2.start()
    p1.join(); p2.join()
    print("2 processes:", round(time.time() - start, 2), "s (real speedup - separate interpreters, no shared GIL)")

## 11. Parallel Computing & Memory Architecture

### Parallel Computing with Python
Python offers two main ways to achieve parallelism:
- **Multithreading** - multiple threads share the same memory space inside one process. Lightweight, but CPU-bound work is limited by the GIL (see above). Best for I/O-bound tasks (network, disk, waiting).
- **Multiprocessing** - separate processes, each with its own interpreter and memory, so no GIL contention. Best for CPU-bound tasks. State must be shared explicitly (`Value`, `Array`, `Manager`, queues, pipes - see section 9).

### Flynn's Taxonomy (Parallel Computing Memory Architecture)
Classifies architectures by how many instruction and data streams run concurrently:

| | Single Data | Multiple Data |
|---|---|---|
| **Single Instruction** | SISD | SIMD |
| **Multiple Instruction** | MISD | MIMD |

- **SISD** (Single Instruction, Single Data) - a classic sequential computer; one instruction stream operates on one data stream at a time. A single Python thread running on one core is SISD.
- **MISD** (Multiple Instruction, Single Data) - multiple instruction streams operate on the same data stream. Rare in practice; mostly used for fault-tolerant/redundant computation in safety-critical systems.
- **SIMD** (Single Instruction, Multiple Data) - one instruction applied to many data elements at once. This is how vectorized NumPy operations and GPUs work.
- **MIMD** (Multiple Instruction, Multiple Data) - independent instruction streams operate on independent data. Multi-core CPUs running separate threads/processes fit here - this is the model Python's `threading`/`multiprocessing` target.

### Memory Organization
- **Shared Memory** - all processing units access the same memory space; communication is just reading/writing shared variables, but needs synchronization (locks) to avoid race conditions. Threads within one process are shared-memory by nature; `multiprocessing.Value`/`Array`/`Manager` simulate shared memory across processes.
- **Distributed Memory** - each processing unit (often a separate machine) has its own private memory; units communicate only by explicitly passing messages over a network (e.g. MPI, or higher-level Python tools like Dask, Ray, Celery). Nothing is implicitly shared - everything must be sent/received.